# Attention Sinks in Graph Transformers — Setup & Baseline

This notebook:
1. Installs GraphGPS and dependencies
2. Trains a baseline GraphGPS model on ZINC-12k
3. Instruments the model to extract per-layer attention matrices and node norms
4. Computes all 7 diagnostic metrics
5. Produces the first layer-wise trajectory plot

## 1. Installation

In [39]:
# Check which GPU is being used by the current server
!nvidia-smi

In [40]:
# Install PyTorch Geometric and dependencies
# Adjust CUDA version if needed (check `nvidia-smi` output)
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install torch-geometric
!pip install ogb  # Open Graph Benchmark (for datasets)
!pip install yacs  # Config system used by GraphGPS
!pip install performer-pytorch  # Optional: for Performer attention variant

In [41]:
# Clone GraphGPS repository
!git clone https://github.com/rampasek/GraphGPS.git
%cd GraphGPS
!pip install -e .

In [42]:
import torch
import torch_geometric
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj, to_dense_batch

print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Load ZINC-12k and inspect

In [ ]:
# Load ZINC-12k (subset=True gives the 12k version)
train_dataset = ZINC(root='data/ZINC', subset=True, split='train')
val_dataset = ZINC(root='data/ZINC', subset=True, split='val')
test_dataset = ZINC(root='data/ZINC', subset=True, split='test')

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"\nExample graph:")
g = train_dataset[0]
print(f"  Nodes: {g.num_nodes}, Edges: {g.num_edges}")
print(f"  Node features shape: {g.x.shape}")
print(f"  Edge attr shape: {g.edge_attr.shape}")
print(f"  Target: {g.y}")

In [ ]:
# Compute dataset statistics
num_nodes = [d.num_nodes for d in train_dataset]
num_edges = [d.num_edges for d in train_dataset]
print(f"Nodes — mean: {np.mean(num_nodes):.1f}, std: {np.std(num_nodes):.1f}, min: {np.min(num_nodes)}, max: {np.max(num_nodes)}")
print(f"Edges — mean: {np.mean(num_edges):.1f}, std: {np.std(num_edges):.1f}, min: {np.min(num_edges)}, max: {np.max(num_edges)}")

## 3. Precompute spectral gaps (lambda_2)

For each graph in **all splits**, compute the second smallest eigenvalue of the normalized Laplacian. This is needed for Experiment 5 (H5: spectral gap predicts sink strength).

We also compute additional spectral properties (spectral radius, number of distinct eigenvalues) and save everything to disk so we never recompute.

In [ ]:
import os
import json
import pickle
from torch_geometric.utils import get_laplacian, to_scipy_sparse_matrix, degree
from scipy.sparse.linalg import eigsh
from tqdm import tqdm

SPECTRAL_CACHE_DIR = 'data/spectral_cache'
os.makedirs(SPECTRAL_CACHE_DIR, exist_ok=True)


def compute_spectral_properties(data):
    """Compute spectral properties of a single graph.
    
    Returns dict with:
      - lambda_2: spectral gap (Fiedler value) of normalized Laplacian
      - spectral_radius: largest eigenvalue of normalized adjacency
      - num_nodes: number of nodes
      - num_edges: number of edges
      - avg_degree: average node degree
      - max_degree: maximum node degree
    """
    n = data.num_nodes
    
    # Edge cases
    if n <= 1:
        return {
            'lambda_2': 0.0, 'spectral_radius': 0.0,
            'num_nodes': n, 'num_edges': data.num_edges,
            'avg_degree': 0.0, 'max_degree': 0,
        }
    
    # Compute normalized Laplacian L_sym = I - D^{-1/2} A D^{-1/2}
    edge_index, edge_weight = get_laplacian(
        data.edge_index, normalization='sym', num_nodes=n
    )
    L = to_scipy_sparse_matrix(edge_index, edge_weight, num_nodes=n)
    
    # Degree stats (from original edges, undirected so divide by 2 for avg)
    deg = degree(data.edge_index[0], num_nodes=n)
    
    # For small graphs (n <= 64), use dense eigendecomposition — exact and fast
    if n <= 64:
        eigenvalues = np.linalg.eigvalsh(L.toarray())
        eigenvalues = np.sort(eigenvalues)
    else:
        # For larger graphs, use sparse ARPACK with shift-invert for robustness
        try:
            # shift-invert mode (sigma=0) is more reliable for smallest eigenvalues
            k = min(6, n - 1)
            eigenvalues = eigsh(L.tocsc(), k=k, sigma=1e-6, which='LM', return_eigenvectors=False)
            eigenvalues = np.sort(eigenvalues)
        except Exception:
            # Final fallback: dense
            eigenvalues = np.linalg.eigvalsh(L.toarray())
            eigenvalues = np.sort(eigenvalues)
    
    # lambda_2 = second smallest eigenvalue (spectral gap / algebraic connectivity)
    # Eigenvalues of L_sym are in [0, 2]. lambda_1 = 0 for connected graphs.
    # Filter out near-zero eigenvalues to find the true gap
    nonzero_eigs = eigenvalues[eigenvalues > 1e-7]
    lambda_2 = float(nonzero_eigs[0]) if len(nonzero_eigs) > 0 else 0.0
    
    # Spectral radius of normalized adjacency = 1 - lambda_min(L_sym) for the 
    # non-trivial part, but more directly: max eigenvalue of D^{-1/2} A D^{-1/2}
    # Since L_sym = I - D^{-1/2} A D^{-1/2}, eigenvalues of adj = 1 - eigenvalues of L
    spectral_radius = float(1.0 - eigenvalues[0])  # largest adjacency eigenvalue
    
    return {
        'lambda_2': lambda_2,
        'spectral_radius': spectral_radius,
        'num_nodes': n,
        'num_edges': data.num_edges,
        'avg_degree': float(deg.mean().item()),
        'max_degree': int(deg.max().item()),
    }


def compute_and_cache_spectral(dataset, split_name):
    """Compute spectral properties for all graphs in a dataset split, with caching."""
    cache_path = os.path.join(SPECTRAL_CACHE_DIR, f'{split_name}_spectral.pkl')
    
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            props = pickle.load(f)
        print(f"  Loaded cached spectral properties for {split_name} ({len(props)} graphs)")
        return props
    
    print(f"  Computing spectral properties for {split_name} ({len(dataset)} graphs)...")
    props = []
    for data in tqdm(dataset, desc=f'  {split_name}'):
        props.append(compute_spectral_properties(data))
    
    with open(cache_path, 'wb') as f:
        pickle.dump(props, f)
    print(f"  Saved to {cache_path}")
    return props


# Compute for all splits
print("Computing spectral properties for all ZINC splits:")
spectral_train = compute_and_cache_spectral(train_dataset, 'train')
spectral_val = compute_and_cache_spectral(val_dataset, 'val')
spectral_test = compute_and_cache_spectral(test_dataset, 'test')

# Extract lambda_2 arrays for easy access
lambda2_train = np.array([s['lambda_2'] for s in spectral_train])
lambda2_val = np.array([s['lambda_2'] for s in spectral_val])
lambda2_test = np.array([s['lambda_2'] for s in spectral_test])

print(f"\nSpectral gap (lambda_2) summary:")
for name, arr in [('Train', lambda2_train), ('Val', lambda2_val), ('Test', lambda2_test)]:
    print(f"  {name:5s} — mean: {arr.mean():.4f}, std: {arr.std():.4f}, "
          f"min: {arr.min():.4f}, max: {arr.max():.4f}")

In [ ]:
# Visualize spectral gap distribution and its relation to graph structure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Histogram of lambda_2
axes[0].hist(lambda2_test, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Spectral gap ($\\lambda_2$)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of spectral gaps (ZINC test)')
axes[0].axvline(np.median(lambda2_test), color='red', linestyle='--', label=f'Median: {np.median(lambda2_test):.3f}')
axes[0].legend()

# 2. lambda_2 vs number of nodes (does graph size affect spectral gap?)
nodes_test = np.array([s['num_nodes'] for s in spectral_test])
axes[1].scatter(nodes_test, lambda2_test, alpha=0.3, s=10, color='steelblue')
axes[1].set_xlabel('Number of nodes')
axes[1].set_ylabel('Spectral gap ($\\lambda_2$)')
axes[1].set_title('Spectral gap vs graph size')

# 3. lambda_2 vs average degree (does connectivity affect spectral gap?)
avg_deg_test = np.array([s['avg_degree'] for s in spectral_test])
axes[2].scatter(avg_deg_test, lambda2_test, alpha=0.3, s=10, color='darkorange')
axes[2].set_xlabel('Average degree')
axes[2].set_ylabel('Spectral gap ($\\lambda_2$)')
axes[2].set_title('Spectral gap vs connectivity')

plt.tight_layout()
plt.savefig('spectral_gap_analysis.pdf', bbox_inches='tight', dpi=150)
plt.show()

# Print correlation stats (useful for later analysis)
from scipy.stats import spearmanr
r_nodes, p_nodes = spearmanr(nodes_test, lambda2_test)
r_deg, p_deg = spearmanr(avg_deg_test, lambda2_test)
print(f"Spearman correlations:")
print(f"  lambda_2 vs num_nodes:  r={r_nodes:.3f}, p={p_nodes:.2e}")
print(f"  lambda_2 vs avg_degree: r={r_deg:.3f}, p={p_deg:.2e}")

## 4. Build GraphGPS model with attention extraction

Standard GraphGPS: GINE (local MPNN) + MultiheadAttention (global) in parallel per layer. No virtual node — we study whether any **real node** develops sink-like attention patterns.

Key instrumentation: we hook into the `MultiheadAttention` module inside each `GPSConv` to extract per-head attention weights, and store node representations at each layer.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, GPSConv, global_add_pool
from torch_geometric.utils import to_dense_batch


class InstrumentedGPS(nn.Module):
    """GraphGPS model instrumented to extract attention weights and node representations.
    
    Matches the GraphGPS ZINC config: GINE + Transformer, RWSE positional encoding,
    batch norm, 3 post-MP layers. No virtual node.
    """
    
    def __init__(
        self,
        num_node_types=28,
        num_edge_types=4,
        hidden_dim=64,
        num_layers=10,
        num_heads=4,
        attn_dropout=0.5,
        dropout=0.0,
        pe_dim=20,          # RWSE walk length (0 = no PE)
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.pe_dim = pe_dim
        
        # Node embedding: atom type -> (hidden_dim - pe_dim)
        # PE gets its own linear projection -> pe_dim
        # Concatenated: hidden_dim total
        self.node_emb = nn.Embedding(num_node_types, hidden_dim - pe_dim if pe_dim > 0 else hidden_dim)
        self.edge_emb = nn.Embedding(num_edge_types, hidden_dim)
        
        # RWSE projection: raw PE -> pe_dim (with batch norm, matching GraphGPS)
        if pe_dim > 0:
            self.pe_encoder = nn.Sequential(
                nn.BatchNorm1d(pe_dim),
                nn.Linear(pe_dim, pe_dim),
                nn.ReLU(),
                nn.Linear(pe_dim, pe_dim),
            )
        
        # GPS layers
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            gine_nn = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            local_model = GINEConv(gine_nn, edge_dim=hidden_dim)
            
            gps_layer = GPSConv(
                channels=hidden_dim,
                conv=local_model,
                heads=num_heads,
                dropout=dropout,
                norm='batch_norm',
                attn_type='multihead',
                attn_kwargs={'dropout': attn_dropout},
            )
            self.layers.append(gps_layer)
        
        # Output head (3 post-MP layers, matching GraphGPS config)
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        
        # Diagnostics storage
        self.layer_data = []
        self.attn_weights = []
        self._attn_hooks = []
    
    def _register_attn_hooks(self):
        """Register forward hooks on MultiheadAttention inside each GPSConv layer."""
        self._remove_attn_hooks()
        self.attn_weights = [None] * self.num_layers
        
        for layer_idx, gps_layer in enumerate(self.layers):
            for name, module in gps_layer.named_modules():
                if isinstance(module, nn.MultiheadAttention):
                    def make_hook(idx):
                        def hook_fn(module, args, output):
                            if isinstance(output, tuple) and len(output) == 2:
                                self.attn_weights[idx] = output[1].detach().cpu()
                        return hook_fn
                    
                    original_forward = module.forward
                    def patched_forward(orig_fwd=original_forward):
                        def wrapper(*args, **kwargs):
                            kwargs['need_weights'] = True
                            kwargs['average_attn_weights'] = False
                            return orig_fwd(*args, **kwargs)
                        return wrapper
                    module.forward = patched_forward()
                    
                    handle = module.register_forward_hook(make_hook(layer_idx))
                    self._attn_hooks.append(handle)
                    break
    
    def _remove_attn_hooks(self):
        for h in self._attn_hooks:
            h.remove()
        self._attn_hooks = []
        self.attn_weights = []
    
    def forward(self, batch_data, collect_diagnostics=False):
        x = batch_data.x
        edge_index = batch_data.edge_index
        edge_attr = batch_data.edge_attr
        batch = batch_data.batch
        
        # Node embedding
        x = self.node_emb(x.squeeze(-1))
        
        # Add RWSE positional encoding
        if self.pe_dim > 0 and hasattr(batch_data, 'random_walk_pe'):
            pe = self.pe_encoder(batch_data.random_walk_pe)
            x = torch.cat([x, pe], dim=-1)  # (num_nodes, hidden_dim)
        elif self.pe_dim > 0:
            # Pad if PE missing (shouldn't happen with correct data loading)
            x = F.pad(x, (0, self.pe_dim))
        
        # Edge embedding
        edge_attr = self.edge_emb(edge_attr.squeeze(-1))
        
        # Record input
        if collect_diagnostics:
            self.layer_data = []
            self.layer_data.append({
                'h': x.detach().cpu(),
                'batch': batch.detach().cpu(),
            })
        
        # GPS layers
        for layer_idx, layer in enumerate(self.layers):
            x = layer(x, edge_index, batch, edge_attr=edge_attr)
            
            if collect_diagnostics:
                self.layer_data.append({
                    'h': x.detach().cpu(),
                    'batch': batch.detach().cpu(),
                })
        
        # Pool and predict
        graph_emb = global_add_pool(x, batch)
        return self.output_head(graph_emb).squeeze(-1)


print("InstrumentedGPS model defined (with RWSE PE, no VNode).")

## 5. Diagnostic metric functions

Metrics adapted for the no-VNode setting. We now measure:
- **Per-node sink score/rate**: which real nodes absorb disproportionate attention?
- **Node norm statistics**: do any nodes develop massive activations?
- **Matrix entropy & anisotropy**: representational compression
- **Dirichlet energy**: over-smoothing
- **Max norm ratio**: ratio of max-norm node to mean norm (replaces VNode ratio)

In [ ]:
def compute_sink_scores(attn_matrix, epsilon=0.3):
    """Compute per-node sink scores from an attention matrix.
    
    attn_matrix: (num_heads, seq_len, seq_len) — attention weights for one graph.
                 attn_matrix[h, i, j] = how much node i attends to node j in head h.
    epsilon: threshold for sink detection.
    
    Returns dict with:
      - sink_score_per_node: (seq_len,) mean attention received by each node, averaged over heads and query nodes
      - sink_rate_per_node: (seq_len,) fraction of heads where the node's avg received attention > epsilon
      - max_sink_score: scalar, highest sink score across all nodes
      - max_sink_node: int, which node has the highest sink score
    """
    H, N, _ = attn_matrix.shape
    
    # sink_score[h, j] = mean attention node j receives across all query nodes i, in head h
    per_head_sink = attn_matrix.mean(dim=1)  # (H, N) — average over query dimension
    
    # Average across heads
    sink_score_per_node = per_head_sink.mean(dim=0)  # (N,)
    
    # Sink rate: fraction of heads where node j receives avg attention > epsilon
    sink_rate_per_node = (per_head_sink > epsilon).float().mean(dim=0)  # (N,)
    
    max_sink_score = sink_score_per_node.max().item()
    max_sink_node = sink_score_per_node.argmax().item()
    
    return {
        'sink_score_per_node': sink_score_per_node,
        'sink_rate_per_node': sink_rate_per_node,
        'max_sink_score': max_sink_score,
        'max_sink_node': max_sink_node,
        'overall_sink_rate': sink_rate_per_node.max().item(),  # strongest sink in this graph
    }


def compute_norm_stats(H):
    """Compute node norm statistics for a single graph.
    H: (num_nodes, hidden_dim)
    """
    norms = torch.norm(H, dim=-1)  # (num_nodes,)
    max_norm = norms.max().item()
    mean_norm = norms.mean().item()
    max_norm_node = norms.argmax().item()
    
    # Ratio of max-norm node to mean — analogous to VNode norm ratio
    # but now for the most extreme real node
    max_to_mean_ratio = (max_norm ** 2) / (mean_norm ** 2) if mean_norm > 1e-10 else 0.0
    
    return {
        'max_norm': max_norm,
        'mean_norm': mean_norm,
        'max_norm_node': max_norm_node,
        'max_to_mean_ratio': max_to_mean_ratio,
        'norm_std': norms.std().item(),
    }


def compute_matrix_entropy(H):
    """Compute normalized matrix-based entropy H(X) in [0, 1]."""
    S = torch.linalg.svdvals(H.float())
    S_sq = S.pow(2)
    total = S_sq.sum()
    if total < 1e-10:
        return 0.0
    p = S_sq / total
    p = p[p > 1e-10]
    entropy = -(p * torch.log(p)).sum().item()
    max_entropy = np.log(len(S))
    return entropy / max_entropy if max_entropy > 0 else 0.0


def compute_anisotropy(H):
    """Compute anisotropy p_1 = sigma_1^2 / ||X||_F^2."""
    S = torch.linalg.svdvals(H.float())
    S_sq = S.pow(2)
    total = S_sq.sum()
    if total < 1e-10:
        return 1.0
    return (S_sq[0] / total).item()


def compute_dirichlet_energy(H, edge_index, num_nodes):
    """Compute unnormalized Dirichlet energy over graph edges."""
    src, dst = edge_index
    valid = (src < num_nodes) & (dst < num_nodes)
    src, dst = src[valid], dst[valid]
    if src.numel() == 0:
        return 0.0
    diff = H[src] - H[dst]
    return diff.pow(2).sum().item()


def compute_all_metrics(H, edge_index, num_nodes, attn_matrix=None):
    """Compute all metrics for a single graph at a single layer.
    
    H: (num_nodes, hidden_dim)
    edge_index: (2, num_edges) 
    num_nodes: int
    attn_matrix: (num_heads, num_nodes, num_nodes) or None
    """
    norm_stats = compute_norm_stats(H)
    
    result = {
        'matrix_entropy': compute_matrix_entropy(H),
        'anisotropy': compute_anisotropy(H),
        'dirichlet_energy': compute_dirichlet_energy(H, edge_index, num_nodes),
        'max_norm': norm_stats['max_norm'],
        'mean_norm': norm_stats['mean_norm'],
        'max_to_mean_ratio': norm_stats['max_to_mean_ratio'],
        'norm_std': norm_stats['norm_std'],
        'max_norm_node': norm_stats['max_norm_node'],
    }
    
    if attn_matrix is not None:
        sink_stats = compute_sink_scores(attn_matrix)
        result['max_sink_score'] = sink_stats['max_sink_score']
        result['overall_sink_rate'] = sink_stats['overall_sink_rate']
        result['max_sink_node'] = sink_stats['max_sink_node']
    
    return result


print("Metric functions defined (no-VNode version with attention sink detection).")

## 6. Training loop

In [ ]:
from torch_geometric.transforms import AddRandomWalkPE

# Use `transform` (applied at load time) NOT `pre_transform` (applied once at processing time).
# pre_transform silently skips if dataset is already cached — this caused the PE=MISSING bug.
pe_transform = AddRandomWalkPE(walk_length=20, attr_name='random_walk_pe')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Verify PE is present
sample = next(iter(train_loader))
has_pe = hasattr(sample, 'random_walk_pe')
if has_pe:
    print(f"RWSE loaded: shape = {sample.random_walk_pe.shape}")
else:
    # Apply transform manually to each dataset
    print("PE not found via DataLoader — applying transform to datasets directly...")
    train_dataset.transform = pe_transform
    val_dataset.transform = pe_transform
    test_dataset.transform = pe_transform
    
    # Rebuild loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    sample = next(iter(train_loader))
    print(f"RWSE after manual transform: present={hasattr(sample, 'random_walk_pe')}, "
          f"shape={sample.random_walk_pe.shape if hasattr(sample, 'random_walk_pe') else 'MISSING'}")

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = InstrumentedGPS(
    num_node_types=28,
    num_edge_types=4,
    hidden_dim=64,
    num_layers=10,
    num_heads=4,
    attn_dropout=0.5,
    dropout=0.0,
    pe_dim=20,
).to(device)

NUM_EPOCHS = 2000
WARMUP_EPOCHS = 50
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)

# Cosine annealing with linear warmup (matching GraphGPS config)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS]
)

criterion = nn.L1Loss()  # MAE

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")
print(f"Device: {device}")
print(f"Training for {NUM_EPOCHS} epochs ({WARMUP_EPOCHS} warmup)")

# Verify PE flows through the model
model.eval()
with torch.no_grad():
    test_batch = next(iter(train_loader)).to(device)
    test_out = model(test_batch)
    print(f"Forward pass OK — output shape: {test_out.shape}")
    pe_used = hasattr(test_batch, 'random_walk_pe')
    print(f"PE used in this batch: {pe_used}")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch, collect_diagnostics=False)
        loss = criterion(pred, batch.y.float())
        loss.backward()
        # Gradient clipping — matches GraphGPS config (clip_grad_norm: True)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch, collect_diagnostics=False)
        loss = criterion(pred, batch.y.float())
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

In [ ]:
from tqdm.auto import tqdm as tqdm_auto

# Train the baseline model
best_val_mae = float('inf')
train_losses = []
val_losses = []

pbar = tqdm_auto(range(1, NUM_EPOCHS + 1), desc='Training')
for epoch in pbar:
    train_mae = train_epoch(model, train_loader, optimizer, criterion, device)
    val_mae = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()
    
    train_losses.append(train_mae)
    val_losses.append(val_mae)
    
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), 'best_model.pt')
    
    pbar.set_postfix({
        'train': f'{train_mae:.4f}',
        'val': f'{val_mae:.4f}',
        'best': f'{best_val_mae:.4f}',
    })

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
test_mae = eval_epoch(model, test_loader, criterion, device)
print(f"\nFinal — Best Val MAE: {best_val_mae:.4f}, Test MAE: {test_mae:.4f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(train_losses, label='Train MAE')
ax.plot(val_losses, label='Val MAE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Training Curves — Baseline GraphGPS on ZINC')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Diagnostic sweep — Extract per-layer metrics and attention weights

Run the trained model on the test set with attention hooks enabled. For each graph at each layer, compute representation metrics and attention sink scores.

In [ ]:
@torch.no_grad()
def run_diagnostics(model, loader, device, max_graphs=200):
    """Run diagnostic sweep on real nodes (no VNode).
    
    Extracts per-layer: node representations, attention weights.
    Computes per-graph: sink scores, norm stats, entropy, anisotropy, Dirichlet energy.
    
    Returns: dict mapping layer_idx -> list of per-graph metric dicts.
    """
    model.eval()
    model._register_attn_hooks()  # Enable attention weight capture
    
    num_layers = model.num_layers
    all_metrics = {l: [] for l in range(num_layers + 1)}  # +1 for input layer (no attn)
    
    graphs_processed = 0
    attn_captured_count = 0  # Debug counter
    
    for batch in tqdm(loader, desc='Diagnostics'):
        if graphs_processed >= max_graphs:
            break
        
        batch = batch.to(device)
        _ = model(batch, collect_diagnostics=True)
        
        # Debug: check if attention hooks captured anything (first batch only)
        if graphs_processed == 0:
            for li in range(num_layers):
                aw = model.attn_weights[li]
                if aw is not None:
                    print(f"  [DEBUG] Layer {li} attn shape: {aw.shape}")
                else:
                    print(f"  [DEBUG] Layer {li} attn: None")
        
        # Get batch info for splitting graphs
        batch_ids = model.layer_data[0]['batch']
        unique_graphs = batch_ids.unique()
        
        for g_idx_in_batch, g_id in enumerate(unique_graphs):
            if graphs_processed >= max_graphs:
                break
            
            graph_mask = (batch_ids == g_id)
            num_nodes_g = graph_mask.sum().item()
            
            for layer_idx in range(num_layers + 1):
                H_graph = model.layer_data[layer_idx]['h'][graph_mask]
                
                # Get attention for this graph at this layer (not available for input layer)
                attn_g = None
                if layer_idx > 0 and model.attn_weights[layer_idx - 1] is not None:
                    attn_full = model.attn_weights[layer_idx - 1]
                    # Shape: (num_graphs_in_batch, num_heads, max_n, max_n)
                    if g_idx_in_batch < attn_full.size(0):
                        attn_g = attn_full[g_idx_in_batch, :, :num_nodes_g, :num_nodes_g]
                        if graphs_processed == 0 and layer_idx == 1:
                            attn_captured_count += 1
                
                metrics = compute_all_metrics(
                    H_graph, batch.edge_index.cpu(), num_nodes_g, attn_g
                )
                all_metrics[layer_idx].append(metrics)
            
            graphs_processed += 1
    
    model._remove_attn_hooks()
    
    # Summary
    print(f"\nAttention capture summary:")
    print(f"  Graphs processed: {graphs_processed}")
    # Check how many graphs at layer 1 have sink scores
    has_sink = sum(1 for m in all_metrics[1] if 'max_sink_score' in m)
    print(f"  Graphs with attention data at layer 1: {has_sink}/{len(all_metrics[1])}")
    
    return all_metrics

print("Running diagnostic sweep on test set...")
metrics = run_diagnostics(model, test_loader, device, max_graphs=200)
print(f"Done. Collected metrics for {len(metrics[0])} graphs across {len(metrics)} layers.")

## 8. Plot layer-wise trajectories (Figure 1 — the main result)

In [ ]:
def aggregate_metrics(metrics_dict):
    """Aggregate per-graph metrics into mean +/- std per layer."""
    layers = sorted(metrics_dict.keys())
    
    # Collect metric names from ALL layers (not just layer 0).
    # Layer 0 is the input — it has no attention data, so sink metrics are missing there.
    # We need to find keys from layers that DO have attention (layers >= 1).
    all_keys = set()
    for l in layers:
        for m in metrics_dict[l]:
            for k, v in m.items():
                if v is not None and not isinstance(v, (list, tuple)):
                    all_keys.add(k)
    metric_names = sorted(all_keys)
    
    result = {}
    for name in metric_names:
        means, stds = [], []
        for l in layers:
            values = [m[name] for m in metrics_dict[l] if m.get(name) is not None]
            if values:
                means.append(np.mean(values))
                stds.append(np.std(values))
            else:
                means.append(np.nan)
                stds.append(np.nan)
        result[name] = {'mean': np.array(means), 'std': np.array(stds)}
    
    return result, layers


agg, layers = aggregate_metrics(metrics)

# Check which metrics were found
print(f"Metrics discovered: {list(agg.keys())}")
print(f"Layers: {layers}")

# Plot the main diagnostic figure — 6 panels for no-VNode analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Layer-wise Diagnostics — GraphGPS on ZINC (no VNode)', fontsize=14)

plot_configs = [
    ('max_sink_score', 'Max Sink Score (any node)', 'tab:red'),
    ('overall_sink_rate', 'Overall Sink Rate', 'tab:orange'),
    ('matrix_entropy', 'Matrix Entropy H(X) (normalized)', 'tab:blue'),
    ('anisotropy', 'Anisotropy p_1', 'tab:green'),
    ('dirichlet_energy', 'Dirichlet Energy', 'tab:purple'),
    ('max_to_mean_ratio', 'Max/Mean Norm Ratio', 'tab:brown'),
]

for ax, (metric_name, title, color) in zip(axes.flat, plot_configs):
    if metric_name in agg:
        mean = agg[metric_name]['mean']
        std = agg[metric_name]['std']
        # Filter out NaN (e.g., layer 0 has no attention data)
        valid = ~np.isnan(mean)
        valid_layers = np.array(layers)[valid]
        ax.plot(valid_layers, mean[valid], color=color, linewidth=2, marker='o', markersize=3)
        ax.fill_between(valid_layers, (mean - std)[valid], (mean + std)[valid], alpha=0.2, color=color)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, color='red')
    ax.set_xlabel('Layer')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figure1_layer_diagnostics.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: figure1_layer_diagnostics.pdf")

## 9. Sanity check: are there attention sinks on real nodes?

Key questions:
- Does any node consistently receive disproportionate attention? (sink score)
- Do any nodes develop larger norms than others? (norm ratio)
- Is there representational compression? (entropy, anisotropy)
- Is there over-smoothing? (Dirichlet energy)

In [ ]:
print("=== Key Results ===\n")

# Attention sinks
if 'max_sink_score' in agg:
    max_ss = np.nanmax(agg['max_sink_score']['mean'])
    max_ss_layer = layers[np.nanargmax(agg['max_sink_score']['mean'])]
    print(f"Attention sinks:")
    print(f"  Max sink score = {max_ss:.4f} at layer {max_ss_layer}")
    print(f"  (Uniform attention would give 1/n ≈ {1/23:.4f} for avg ZINC graph)")
    if max_ss > 3 * (1/23):
        print(f"  -> Some nodes receive {max_ss * 23:.1f}x more attention than uniform")
    else:
        print(f"  -> Attention is relatively diffuse (no strong sinks)")

if 'overall_sink_rate' in agg:
    max_sr = np.nanmax(agg['overall_sink_rate']['mean'])
    print(f"  Max sink rate (eps=0.3) = {max_sr:.4f}")

# Norm concentration
print()
if 'max_to_mean_ratio' in agg:
    max_ratio = np.nanmax(agg['max_to_mean_ratio']['mean'])
    max_ratio_layer = layers[np.nanargmax(agg['max_to_mean_ratio']['mean'])]
    print(f"Norm concentration:")
    print(f"  Max norm²/mean norm² ratio = {max_ratio:.2f} at layer {max_ratio_layer}")
    if max_ratio > 10:
        print(f"  -> Some node develops massive activations (ratio > 10)")
    else:
        print(f"  -> Norms are relatively uniform (no massive activations)")

# Compression
print()
min_entropy = np.nanmin(agg['matrix_entropy']['mean'])
min_entropy_layer = layers[np.nanargmin(agg['matrix_entropy']['mean'])]
max_aniso = np.nanmax(agg['anisotropy']['mean'])
print(f"Compression:")
print(f"  Min matrix entropy = {min_entropy:.4f} at layer {min_entropy_layer}")
print(f"  Max anisotropy p_1 = {max_aniso:.4f}")
if min_entropy < 0.5:
    print(f"  -> Compression valley detected")
else:
    print(f"  -> No significant compression")

# Over-smoothing
print()
de_first = agg['dirichlet_energy']['mean'][0]
de_last = agg['dirichlet_energy']['mean'][-1]
print(f"Over-smoothing:")
print(f"  Dirichlet energy: layer 0 = {de_first:.1f}, layer {layers[-1]} = {de_last:.1f}")
print(f"  Ratio last/first = {de_last/de_first:.4f}")
if de_last < de_first * 0.1:
    print(f"  -> Strong over-smoothing (>90% energy reduction)")
elif de_last < de_first * 0.5:
    print(f"  -> Moderate over-smoothing")
else:
    print(f"  -> Mild or no over-smoothing")

## Next steps

Based on these baseline results:
1. **If sinks found on real nodes** → which nodes? Correlate with degree, spectral properties. Run ablations (depth, MPNN on/off).
2. **If no sinks found** → this is also a finding: Graph Transformers handle over-mixing differently from LLMs (no canonical sink target). Investigate why: small graphs? shallow depth? MPNN provides mixing control?
3. **Either way** → run perturbation experiment (does removing high-attention nodes increase perturbation spread?) and spectral gap correlation (H5).